In [1]:
import os, sys, platform, yaml, re
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, FloatType, StringType
from pathlib import Path
import matplotlib.pyplot as plt
import hmac
import hashlib

In [2]:
conf_path = str(Path.cwd() / "config" / "ETL_config.yaml")
with open(conf_path, "r") as f:
    CFG = yaml.safe_load(f)

IS_WIN          = platform.system() == "Windows"
CSV_DIR         = CFG["paths"]["csv_base_dir"]["windows" if IS_WIN else "linux"]
PG_URL          = CFG["postgres"]["url"]
PG_USER         = CFG["postgres"]["user"]
PG_PASS         = CFG["postgres"]["pass"]
PG_SCHEMA       = CFG["postgres"]["schema_out"]["schema_name"]
PG_TABLE1       = CFG["postgres"]["schema_out"]["table1"]
PG_TABLE2       = CFG["postgres"]["schema_out"]["table2"]
PG_TABLE3       = CFG["postgres"]["schema_out"]["table3"]
FILES           = CFG["csv"]["files"]
NUM_PARTITIONS  = CFG["csv"]["num_partitions"]
JDBC_BATCHSIZE  = CFG["postgres"]["batchsize"]
JDBC_FETCHSIZE  = CFG["postgres"]["fetchsize"]
SPARK_LOCAL_DIR = CFG["spark"]["local_dirs"]["windows" if IS_WIN else "linux"]
NEO4J_URI  = CFG["neo4j"]["uri"]          # "bolt://localhost:7687"
NEO4J_USER = CFG["neo4j"]["user"]
NEO4J_PASS = CFG["neo4j"]["pass"]
NEO4J_DB   = CFG["neo4j"]["database"]

opts = {
    "url": NEO4J_URI,
    "authentication.type": "basic",
    "authentication.basic.username": NEO4J_USER,
    "authentication.basic.password": NEO4J_PASS,
    "database": NEO4J_DB,
}

os.environ["PYSPARK_PYTHON"] = sys.executable       # usa el Python del kernel actual
os.environ["JAVA_HOME"] = os.environ.get("JAVA_HOME", "/usr/lib/jvm/java-17-openjdk-amd64") #Chequear versión Windows después
Path(SPARK_LOCAL_DIR).mkdir(parents=True, exist_ok=True)

In [3]:
builder = (
    SparkSession.builder
    .master("local[*]")  
    .appName(CFG["spark"]["app_name"])
    .config("spark.sql.shuffle.partitions", str(CFG["spark"]["shuffle_partitions"]))
    .config("spark.driver.memory", CFG["spark"]["driver_memory"])
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.local.dir", SPARK_LOCAL_DIR)   
    .config(
        "spark.jars.packages",
        ",".join([
            "org.postgresql:postgresql:42.7.4",
            "org.neo4j:neo4j-connector-apache-spark_2.12:5.3.10_for_spark_3"
        ])
    )
)

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")

jdbc_props = {
        "user": PG_USER,
        "password": PG_PASS,
        "driver": "org.postgresql.Driver",
        "fetchsize": str(JDBC_FETCHSIZE)
    }

25/12/03 09:29:07 WARN Utils: Your hostname, AsusMare resolves to a loopback address: 127.0.1.1; using 192.168.1.12 instead (on interface wlp2s0)
25/12/03 09:29:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/opt/spark-3.5.3-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/felpipe/.ivy2/cache
The jars for the packages stored in: /home/felpipe/.ivy2/jars
org.postgresql#postgresql added as a dependency
org.neo4j#neo4j-connector-apache-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-bbdd6427-da5a-4508-a281-26d28fb4a6b1;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.4 in central
	found org.checkerframework#checker-qual;3.42.0 in central
	found org.neo4j#neo4j-connector-apache-spark_2.12;5.3.10_for_spark_3 in central
	found org.neo4j#neo4j-connector-apache-spark_2.12_common;5.3.10_for_spark_3 in central
	found org.neo4j#caniuse-core;1.3.0 in local-m2-cache
	found org.neo4j#caniuse-api;1.3.0 in local-m2-cache
	found org.jetbrains.kotlin#kotlin-stdlib;2.1.20 in local-m2-cache
	found org.jetbrains#annotations;13.0 in local-m2-cache
	found org.neo4j#caniuse-neo4j-detection;1.3.0 in local-m2-cache
	found org.neo4j.driver#neo4j-java-driver-slim;4.4.21 in local-m2-cache
	

# Anonimización

In [4]:
df_acc = (
    spark.read.format("jdbc")
    .option("url", PG_URL)
    .option("dbtable", f"{PG_SCHEMA}.{PG_TABLE1}")  
    .option("user", PG_USER)
    .option("password", PG_PASS)
    .option("driver", "org.postgresql.Driver")
    .option("partitionColumn", "account")
    .option("lowerBound", "1")
    .option("upperBound", "10000000")
    .option("numPartitions", "6")
    .option("fetchsize", str(JDBC_FETCHSIZE))
    .load()
)

df_txs = (
    spark.read.format("jdbc")
    .option("url", PG_URL)
    .option("dbtable", f"{PG_SCHEMA}.{PG_TABLE2}")  
    .option("user", PG_USER)
    .option("password", PG_PASS)
    .option("driver", "org.postgresql.Driver")
    .option("partitionColumn", "id")
    .option("lowerBound", "1")
    .option("upperBound", "10000000")
    .option("numPartitions", "6")
    .option("fetchsize", str(JDBC_FETCHSIZE))
    .load()
)

In [5]:
HMAC_KEY = b"UNA_CLAVE_SECRETA_LARGA_AQUI" # 32 bytes generada con CPRING

def hmac_sha256(value: str) -> str:
    if value is None:
        return None
    elif type(value) == int:
        value = str(value)

    #Asegurar que value venga ya normalizado como string
    value_bytes = value.encode("utf-8")
    return hmac.new(HMAC_KEY, value_bytes, hashlib.sha256).hexdigest()

hmac_udf = F.udf(hmac_sha256, StringType())

In [9]:
df_hash = (
    df_acc
    .select(F.col("account").alias("cta"))
    .withColumn("cta_tok", hmac_udf(F.col("cta")))
)

(df_hash.write
 .format("jdbc")
 .option("url", PG_URL)
 .option("dbtable", f"{PG_SCHEMA}.cta_hash")
 .option("user", PG_USER)
 .option("password", PG_PASS)
 .option("driver", "org.postgresql.Driver")
 .option("batchsize", str(JDBC_BATCHSIZE))
 .option("truncate", "true") 
 .mode("overwrite")  # o 'append'
 .save())

In [40]:
df_clientes = (
    df_acc
    .select(F.col("account").alias("cta"), F.col("location").alias("bco_cta"))
    .withColumn("cta", hmac_udf(F.col("cta")))
)

df_entrada = (
    df_txs
    .select(
        F.col("date_time").alias("event_date"),
        F.col("amount").alias("mto_trf"),
        F.col("sender_account").alias("cta_ori"),
        F.col("receiver_account").alias("cta_dst"),
        F.col("received_currency").alias("cod_mon")
    )
    .withColumn("cta_ori", hmac_udf(F.col("cta_ori")))
    .withColumn("cta_dst", hmac_udf(F.col("cta_dst")))
    .join(
        df_clientes,
        F.col("cta_dst") == F.col("cta"),
        "inner"
    )
    .filter(F.col("bco_cta") == F.lit("UK"))
).select(
    F.col("event_date"),
    F.col("mto_trf"),
    F.col("cta_ori"),
    F.col("cta_dst"),
    F.col("cod_mon"),
    F.col("bco_cta").alias("bco_dst")
)

df_salida = (
    df_txs
    .select(
        F.col("date_time").alias("event_date"),
        F.col("amount").alias("mto_trf"),
        F.col("sender_account").alias("cta_ori"),
        F.col("receiver_account").alias("cta_dst"),
        F.col("received_currency").alias("cod_mon")
    )
    .withColumn("cta_ori", hmac_udf(F.col("cta_ori")))
    .withColumn("cta_dst", hmac_udf(F.col("cta_dst")))
    .join(
        df_clientes,
        F.col("cta_ori") == F.col("cta"),
        "inner"
    )
    .filter(F.col("bco_cta") == F.lit("UK"))
).select(
    F.col("event_date"),
    F.col("mto_trf"),
    F.col("cta_ori"),
    F.col("cta_dst"),
    F.col("cod_mon"),
    F.col("bco_cta").alias("bco_ori")
)

In [42]:
(df_clientes.write
 .format("jdbc")
 .option("url", PG_URL)
 .option("dbtable", f"{PG_SCHEMA}.cc")
 .option("user", PG_USER)
 .option("password", PG_PASS)
 .option("driver", "org.postgresql.Driver")
 .option("batchsize", str(JDBC_BATCHSIZE))
 .option("truncate", "true") 
 .mode("overwrite")  # o 'append'
 .save())

In [43]:
(df_entrada.write
 .format("jdbc")
 .option("url", PG_URL)
 .option("dbtable", f"{PG_SCHEMA}.txe")
 .option("user", PG_USER)
 .option("password", PG_PASS)
 .option("driver", "org.postgresql.Driver")
 .option("batchsize", str(JDBC_BATCHSIZE))
 .option("truncate", "true") 
 .mode("overwrite")  # o 'append'
 .save())

In [44]:
(df_salida.write
 .format("jdbc")
 .option("url", PG_URL)
 .option("dbtable", f"{PG_SCHEMA}.txs")
 .option("user", PG_USER)
 .option("password", PG_PASS)
 .option("driver", "org.postgresql.Driver")
 .option("batchsize", str(JDBC_BATCHSIZE))
 .option("truncate", "true") 
 .mode("overwrite")  # o 'append'
 .save())

# Ego-Graph

In [45]:
del df_clientes, df_entrada, df_salida #Ahora los consultamos directamente de PostGres

In [48]:
df_clientes = (
    spark.read.format("jdbc")
    .option("url", PG_URL)
    .option("dbtable", f"{PG_SCHEMA}.cc")  
    .option("user", PG_USER)
    .option("password", PG_PASS)
    .option("driver", "org.postgresql.Driver")
    #.option("partitionColumn", "cta")
    #.option("lowerBound", "1")
    #.option("upperBound", "10000000")
    #.option("numPartitions", "6")
    .option("fetchsize", str(JDBC_FETCHSIZE))
    .load()
)

In [54]:
df_entrada = (
    spark.read.format("jdbc")
    .option("url", PG_URL)
    .option("dbtable", f"{PG_SCHEMA}.txe")  
    .option("user", PG_USER)
    .option("password", PG_PASS)
    .option("driver", "org.postgresql.Driver")
    .option("partitionColumn", "event_date")
    .option("lowerBound", "2015-01-01 00:00:00")
    .option("upperBound", "2025-01-01 00:00:00")
    .option("numPartitions", "6")
    .option("fetchsize", str(JDBC_FETCHSIZE))
    .load()
)

In [56]:
df_salida = (
    spark.read.format("jdbc")
    .option("url", PG_URL)
    .option("dbtable", f"{PG_SCHEMA}.txs")  
    .option("user", PG_USER)
    .option("password", PG_PASS)
    .option("driver", "org.postgresql.Driver")
    .option("partitionColumn", "event_date")
    .option("lowerBound", "2015-01-01 00:00:00")
    .option("upperBound", "2025-01-01 00:00:00")
    .option("numPartitions", "6")
    .option("fetchsize", str(JDBC_FETCHSIZE))
    .load()
)

In [57]:
df_salida.limit(10).toPandas()

,event_date,mto_trf,cta_ori,cta_dst,cod_mon,bco_ori
0,2023-02-21 10:50:20,10369.40,0011c1da15fb6054ad9fa1433fd28dc66b94ae61fe05d9...,39be6ea6f76bd87c83f6f108f034f4cd47dcd2d5034d97...,UK pounds,UK
1,2023-02-21 18:46:44,10328.49,0011c1da15fb6054ad9fa1433fd28dc66b94ae61fe05d9...,39be6ea6f76bd87c83f6f108f034f4cd47dcd2d5034d97...,UK pounds,UK
2,2023-02-21 13:17:36,10340.79,0011c1da15fb6054ad9fa1433fd28dc66b94ae61fe05d9...,39be6ea6f76bd87c83f6f108f034f4cd47dcd2d5034d97...,UK pounds,UK
3,2023-02-21 10:17:53,10368.69,0011c1da15fb6054ad9fa1433fd28dc66b94ae61fe05d9...,39be6ea6f76bd87c83f6f108f034f4cd47dcd2d5034d97...,UK pounds,UK
4,2023-02-21 21:10:43,10355.35,0011c1da15fb6054ad9fa1433fd28dc66b94ae61fe05d9...,39be6ea6f76bd87c83f6f108f034f4cd47dcd2d5034d97...,UK pounds,UK
5,2023-02-21 16:21:42,10402.97,0011c1da15fb6054ad9fa1433fd28dc66b94ae61fe05d9...,39be6ea6f76bd87c83f6f108f034f4cd47dcd2d5034d97...,UK pounds,UK
6,2023-02-21 12:12:37,10341.68,0011c1da15fb6054ad9fa1433fd28dc66b94ae61fe05d9...,39be6ea6f76bd87c83f6f108f034f4cd47dcd2d5034d97...,UK pounds,UK
7,2023-02-21 16:36:01,10280.75,0011c1da15fb6054ad9fa1433fd28dc66b94ae61fe05d9...,39be6ea6f76bd87c83f6f108f034f4cd47dcd2d5034d97...,UK pounds,UK
8,2023-02-21 19:25:28,10347.04,0011c1da15fb6054ad9fa1433fd28dc66b94ae61fe05d9...,39be6ea6f76bd87c83f6f108f034f4cd47dcd2d5034d97...,UK pounds,UK
9,2023-02-21 14:27:38,10302.50,0011c1da15fb6054ad9fa1433fd28dc66b94ae61fe05d9...,39be6ea6f76bd87c83f6f108f034f4cd47dcd2d5034d97...,UK pounds,UK
